# Ngoại kiểm rev5c (29/10/2026) — INCART + SVDB, thêm head_scratch công bằng
Giống hệt ngoại kiểm rev5b (25/10), chỉ thêm `head_scratch_fair` (η 0,05, 300 vòng, cap 1, λ 0, fix; chọn trên DS1 ở E13, protocol 26/10)
và bảng số bản ghi bị hại theo bệnh nhân. Hai CSDL này **không còn mù** — chạy chỉ để bảng đầy đủ, không chọn gì.
- Emb lấy từ cache trên Drive → chỉ vài phút.
- Kiểm tra: các phương pháp cũ phải cho **đúng** số của rev5b (ô cuối tự đối chiếu).
- Kết quả: `planA/06_eval/external_rev5c_20261029/`.

In [ ]:
!pip install -q wfdb
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, json, time, hashlib, platform, datetime
import numpy as np

REPO     = '/content/drive/MyDrive/ECG_FPGA_1DCNN/planA'                             # kho phương án A (= H:\My Drive\ECG_FPGA_1DCNN\planA)
DATA_DIR = '/content/drive/MyDrive/mitdb_extracted/mit-bih-arrhythmia-database-1.0.0' # MIT-BIH gốc (DS1 cho neo, DS2 cho kiểm tra board)
MODEL_DIR = '/content/drive/MyDrive/ecg_processed/improve_v4'                          # F2_rr8_str_s{seed}.keras + _quant.json
EXT_DIR  = '/content/drive/MyDrive/ecg_external_wfdb'                                  # INCART, SVDB (tự tải nếu thiếu)
CACHE    = '/content/drive/MyDrive/ecg_processed/improve_v4/pa_phaseA/ext_int_cache'  # cache emb số nguyên (không đưa vào kho)
OUT      = os.path.join(REPO, '06_eval', 'external_rev5c_20261029')                     # KẾT QUẢ MỚI -> kho
OLD5B    = os.path.join(REPO, '06_eval', 'external_rev5b_20261025')
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)

SRC = os.path.join(REPO, 'src'); sys.path.insert(0, SRC)
REV4 = os.path.join(REPO, '06_eval', 'phaseA_rev5_20261025'); sys.path.insert(0, os.path.join(REV4, 'src'))
HP_FILE    = os.path.join(REPO, '06_eval', 'phaseA_exp4', 'exp4_best_hp_DS1.json')
BOARD_FILE = os.path.join(REPO, '05_board', 'ecg_f2a_phaseA', 'f2a_board_results_100MHz.npz')
OLD_DIR    = os.path.join(REPO, '06_eval', 'external_incart_svdb')                   # kết quả pipeline cũ để đối chiếu
SEEDS   = [1, 2, 3, 4, 5]
METHODS_A = ['none', 'offset_int', 'gradient_anchor']                   # chỉ để kiểm tra khớp board (phương án A)
SEL_FILE = os.path.join(REV4, 'out', 'ds1_selected_rev5b_20261025.json')

def sha256(p, n=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(n), b''):
            h.update(b)
    return h.hexdigest()

INPUTS = {'hw_ref.py': os.path.join(SRC, 'hw_ref.py'), 'pa_adapt.py': os.path.join(SRC, 'pa_adapt.py'),
          'exp4_best_hp_DS1.json': HP_FILE, 'f2a_board_results_100MHz.npz': BOARD_FILE,
          'ds1_selected_rev5b_20261025.json': SEL_FILE, 'rev4_methods.py': os.path.join(REV4, 'src', 'rev4_methods.py'),
          'fair_methods.py': os.path.join(REV4, 'src', 'fair_methods.py'), 'e13_selected.json': os.path.join(REV4, 'out', 'bosung_20261026', 'e13_selected.json')}
for s in SEEDS:
    INPUTS[f'F2_rr8_str_s{s}.keras'] = os.path.join(MODEL_DIR, f'F2_rr8_str_s{s}.keras')
    INPUTS[f'F2_rr8_str_s{s}_quant.json'] = os.path.join(MODEL_DIR, f'F2_rr8_str_s{s}_quant.json')
missing = [k for k, p in INPUTS.items() if not os.path.exists(p)]
assert not missing, f'Thiếu file đầu vào: {missing}'
HASHES = {k: sha256(p) for k, p in INPUTS.items()}
for k, v in HASHES.items():
    print(f'{k:32s} {v[:16]}…')

import hw_ref as hw, pa_adapt as pa
best_hp = json.load(open(HP_FILE))
HP_A = {m: best_hp.get(m, {}) for m in METHODS_A}
import rev4_methods as M, fair_methods  # noqa: F401
SEL = json.load(open(SEL_FILE))['selected']
E13 = json.load(open(INPUTS['e13_selected.json']))
CONF = {'none': ('none', {}), 'head_scratch': ('head_scratch', SEL['head_scratch']['hp']), 'head_scratch_fair': ('head_scratch_fair', E13['S_rob']['hp']), 'head_global': ('head_global', SEL['head_global']['hp']),
        'offset_int': ('offset_int', SEL['offset_int']['hp']), 'gradient_anchor': ('gradient_anchor', SEL['gradient_anchor']['hp']),
        'GA-R': ('GA-S', SEL['GA-R']['hp'])}
METHODS = list(CONF)
print('Cấu hình (S_rob, DS1):', CONF)

In [ ]:
import wfdb
from scipy.ndimage import median_filter
from scipy.signal import butter, filtfilt, resample_poly
from math import gcd

FS_RAW = 360
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124, 201, 203,
       205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212, 213, 214,
       219, 221, 222, 228, 231, 232, 233, 234]
AAMI = {'N': 'N', 'L': 'N', 'R': 'N', 'e': 'N', 'j': 'N',
        'A': 'S', 'a': 'S', 'J': 'S', 'S': 'S',
        'V': 'V', 'E': 'V', 'F': 'F', '/': 'Q', 'f': 'Q', 'Q': 'Q'}
BEAT_SYMBOLS = set(AAMI.keys())
CLASS_NAMES = ['N', 'S', 'V', 'F']
CFG = dict(pre_s=0.9, post_s=1.1, ds=2, filt=True)
N_RR = 8
EXT_DBS = {'incartdb': dict(lead='II',   name='St Petersburg INCART',     n_expected=75),
           'svdb':     dict(lead='ECG1', name='MIT-BIH Supraventricular', n_expected=78)}

def load_record(rec):
    # rec kiểu int -> MIT-BIH (MLII, 360 Hz). rec kiểu 'db/tên' -> bản ghi ngoài, nội suy về 360 Hz.
    if not (isinstance(rec, str) and '/' in rec):
        path = os.path.join(DATA_DIR, str(rec))
        r = wfdb.rdrecord(path)
        ch = r.sig_name.index('MLII') if 'MLII' in r.sig_name else 0
        ann = wfdb.rdann(path, 'atr')
        return r.p_signal[:, ch].astype(np.float64), np.asarray(ann.sample), np.asarray(ann.symbol)
    db, name = rec.split('/', 1)
    path = os.path.join(EXT_DIR, db, name)
    r = wfdb.rdrecord(path)
    lead = EXT_DBS[db]['lead']
    ch = r.sig_name.index(lead) if lead in r.sig_name else 0
    sig = np.nan_to_num(r.p_signal[:, ch].astype(np.float64))
    fs = int(round(r.fs)); g = gcd(FS_RAW, fs)
    if fs != FS_RAW:
        sig = resample_poly(sig, FS_RAW // g, fs // g)
    ann = wfdb.rdann(path, 'atr')
    samples = np.round(np.asarray(ann.sample) * FS_RAW / fs).astype(int)
    return sig, samples, np.asarray(ann.symbol)

def clean_signal(sig, fs=FS_RAW):
    base = median_filter(sig, size=int(0.2 * fs) | 1, mode='nearest')
    base = median_filter(base, size=int(0.6 * fs) | 1, mode='nearest')
    b, a = butter(4, 40.0 / (fs / 2), btype='low')
    return filtfilt(b, a, sig - base)

def extract_record(rec, pre_s=0.9, post_s=1.1, ds=2, filt=True):
    # -> X (n,360) float32, R (n,10) float32, y, rec, T (thời điểm nhịp, giây). Cùng vòng lặp với exp3/exp4.
    sig, samples, symbols = load_record(rec)
    if filt:
        sig = clean_signal(sig)
    fs = FS_RAW / ds
    if ds > 1:
        sig = sig[::ds]
    pre, post = int(round(pre_s * fs)), int(round(post_s * fs))
    is_beat = np.array([s in BEAT_SYMBOLS for s in symbols])
    r_all = samples[is_beat] / ds; sym_all = symbols[is_beat]
    rr = np.diff(r_all) / fs
    X, R, Y, T = [], [], [], []
    for i in range(1, len(r_all) - 1):
        lab = AAMI[sym_all[i]]
        if lab == 'Q':
            continue
        c = int(round(r_all[i]))
        if c - pre < 0 or c + post > len(sig):
            continue
        w = sig[c - pre:c + post]; w = w - np.median(w)
        m = np.max(np.abs(w))
        if m < 1e-9:
            continue
        pre_rr, post_rr = rr[i - 1], rr[i]
        local = np.mean(rr[max(0, i - 10):i]); local30 = np.mean(rr[max(0, i - 30):i])
        med300 = np.median(rr[max(0, i - 300):i])
        X.append((w / m).astype(np.float32))
        R.append([pre_rr, post_rr, local, pre_rr / local, post_rr / local, pre_rr / post_rr,
                  pre_rr / local30, post_rr / local30, pre_rr / med300, post_rr / med300])
        Y.append(CLASS_NAMES.index(lab)); T.append(r_all[i] / fs)
    return (np.array(X), np.array(R, dtype=np.float32), np.array(Y), np.array([rec] * len(Y), dtype=object), np.array(T))

def build_dataset(records):
    parts = [extract_record(r, **CFG) for r in records]
    X, R, y, g, T = (np.concatenate([p[k] for p in parts]) for k in range(5))
    g = g.astype(str) if isinstance(records[0], str) else g.astype(int)
    return X, R, y, g, T

def describe(y, name):
    print(f'{name:40s}', dict(zip(CLASS_NAMES, np.bincount(y, minlength=4).tolist())), 'tổng', len(y))

In [ ]:
import tensorflow as tf

def fq(x, f, bits=8):
    s = 2.0 ** f
    return np.clip(np.round(x * s), -2 ** (bits - 1), 2 ** (bits - 1) - 1) / s

def rr_normalize(R):
    return (R[:, :N_RR] / np.array([1.5, 1.5, 1.5, 2, 2, 2, 2, 2], dtype=np.float32)).clip(0, 3.9)

def to_i8(x, f):
    return np.clip(np.round(np.asarray(x, np.float64) * 2 ** f), -128, 127).astype(np.int8)

def q_inputs(X, R, qcfg):
    # Giống hệt exp4 (cell 17) và cell xuất dữ liệu cho board.
    return (to_i8(fq(X.astype(np.float64), qcfg['input']), qcfg['input']),
            to_i8(fq(rr_normalize(R).astype(np.float64), qcfg['rr']), qcfg['rr']))

def _iexact(a):
    # float64 -> int64; chỉ dùng khi mọi giá trị là số nguyên |x| < 2^53 (luôn đúng ở đây: |acc| < 2^26)
    return np.rint(a).astype(np.int64)

def forward_emb_int_fast(beats_q, rr_q, P, qcfg, batch=2048):
    # Cùng thuật toán với hw_ref.forward_emb_int, nhưng nhân ma trận bằng BLAS float64 (chính xác tuyệt đối).
    fin, frr = qcfg['input'], qcfg['rr']
    Pf = {k: (v[0].astype(np.float64), v[1], v[2], v[3]) for k, v in P.items()}
    outs = []
    for s in range(0, len(beats_q), batch):
        x = beats_q[s:s + batch, :, None].astype(np.int64); fx = fin
        for i in (1, 2, 3):
            Wf, fw, b, fb = Pf[f'conv1d_{i}']
            k = Wf.shape[0]; pl = (k - 1) // 2; pr = k - 1 - pl
            xp = np.pad(x, ((0, 0), (pl, pr), (0, 0))).astype(np.float64)
            win = np.lib.stride_tricks.sliding_window_view(xp, k, axis=1)
            fa = fx + fw
            acc = _iexact(np.einsum('nlck,kcf->nlf', win, Wf, optimize=True)) + (b << (fa - fb))
            x = hw._pool(hw._act(acc, fa, qcfg[f'relu_{i}'])); fx = qcfg[f'relu_{i}']
        x = x.reshape(len(x), -1)
        Wf, fw, b, fb = Pf['rr_dense']
        fa = frr + fw
        r = hw._act(_iexact(rr_q[s:s + batch].astype(np.float64) @ Wf) + (b << (fa - fb)), fa, qcfg['rr_dense'])
        Wf, fw, b, fb = Pf['dense_1']
        f1, f2 = qcfg['relu_3'], qcfg['rr_dense']; fm = max(f1, f2); fa = fm + fw
        xx = np.concatenate([x << (fm - f1), r << (fm - f2)], axis=1)
        outs.append(hw._act(_iexact(xx.astype(np.float64) @ Wf) + (b << (fa - fb)), fa, qcfg['dense_1']))
    return np.concatenate(outs)

def load_seed(seed):
    # Trọng số số nguyên + lớp đầu ra toàn cục ĐÚNG như nạp vào head_w/head_b của IP (giống exp4 cell 19).
    model = tf.keras.models.load_model(os.path.join(MODEL_DIR, f'F2_rr8_str_s{seed}.keras'), compile=False)
    qj = json.load(open(os.path.join(MODEL_DIR, f'F2_rr8_str_s{seed}_quant.json'))); qcfg = qj['qcfg']
    Wd = {l.name: l.get_weights() for l in model.layers if l.get_weights()}
    P = hw.quantize_params(Wd, qcfg)
    wo, fw, bq14 = hw.head_global_int(P, qj['offset'])
    fe = qcfg['dense_1']                       # số bit thập phân của emb (seed 3 = 4; KHÔNG cố định)
    W0, b0 = (wo / 2.0 ** fw).T, bq14 / 2.0 ** 14
    return dict(P=P, qcfg=qcfg, fw=fw, fe=fe, W0=W0, b0=b0, wo=wo, bq14=bq14)

def emb_cached(tag, seed, X, R, S):
    p = os.path.join(CACHE, f'emb_int_{tag}_s{seed}.npz')
    if os.path.exists(p):
        z = np.load(p)
        if z['E'].shape[0] == len(X):
            return z['E'].astype(np.int64)
    t0 = time.time()
    bq, rq = q_inputs(X, R, S['qcfg'])
    E = forward_emb_int_fast(bq, rq, S['P'], S['qcfg'])
    np.savez_compressed(p, E=E.astype(np.uint8))
    print(f'   emb {tag} seed {seed}: {len(X)} nhịp, {time.time() - t0:.0f} s')
    return E

In [ ]:
t0 = time.time()
X1, R1, y1, g1, T1 = build_dataset(DS1)
X2, R2, y2, g2, T2 = build_dataset(DS2)
describe(y1, 'DS1'); describe(y2, 'DS2')
assert len(y1) == 50957 and len(y2) == 49649, 'Số nhịp MIT-BIH khác exp4 -> kiểm tra lại dữ liệu'
cal2 = pa.split_first(T2, g2, 5); ev2 = ~cal2
assert cal2.sum() == 8209 and ev2.sum() == 41440, 'Tách 5 phút khác exp4'
print(f'Trích xuất MIT-BIH: {time.time() - t0:.0f} s')

SEED_DATA, ANCHORS = {}, {}
for seed in SEEDS:
    S = load_seed(seed); SEED_DATA[seed] = S
    # (1) bản nhanh phải trùng từng bit với hw_ref.forward_emb_int
    bq, rq = q_inputs(X2[:1500], R2[:1500], S['qcfg'])
    assert np.array_equal(forward_emb_int_fast(bq, rq, S['P'], S['qcfg']), hw.forward_emb_int(bq, rq, S['P'], S['qcfg'])), \
        f'seed {seed}: bản nhanh KHÁC hw_ref'
    # neo: 25 nhịp/lớp từ DS1, cùng hạt giống như exp4
    ai = pa.sample_anchor_idx(y1, np.ones(len(y1), bool), m=25, seed=seed)
    bq, rq = q_inputs(X1[ai], R1[ai], S['qcfg'])
    ANCHORS[seed] = (forward_emb_int_fast(bq, rq, S['P'], S['qcfg']) / 2.0 ** S['fe'], y1[ai])
    print(f'seed {seed}: bit-exact với hw_ref OK | fe = {S["fe"]} | {len(ai)} nhịp neo')

In [ ]:
# (2) Seed 3 trên DS2 phải trùng TỪNG NHÃN với dự đoán đo trên board KV260 (100 MHz)
S = SEED_DATA[3]
E2 = emb_cached('mitdb_DS2', 3, X2, R2, S) / 2.0 ** S['fe']
lg = hw.logits_int(E2 * 2 ** S['fe'], S['wo'], S['fw'], S['bq14'], f_emb=S['fe'])
acc_all = (lg.argmax(1) == y2).mean()
print(f'none trên toàn DS2 (49 649 nhịp): acc {acc_all:.5f}  (f2_hw_config_seed3: 0.94010)')
board = np.load(BOARD_FILE)
pa.CURRENT_ANCHORS = ANCHORS[3]
for mth in METHODS_A:
    yy, pp, _ = pa.run_dataset_split(E2, y2, g2, cal2, ev2, S['W0'], S['b0'], mth, True, S['fw'], **HP_A[mth])
    nd = int((board[f'pred_{mth}'][ev2] != pp).sum())
    print(f'{mth:16s} {pa.fmt(pa.metrics(yy, pp))} | lệch board: {nd} nhịp')
    assert nd == 0, f'{mth}: lệch board {nd} nhịp -> DỪNG, không chạy dữ liệu ngoài'
print('==> Pipeline số nguyên khớp board 0 nhịp lệch. Tiếp tục với dữ liệu ngoài.')

In [ ]:
def complete_records(d):
    heas = {f[:-4] for f in os.listdir(d) if f.endswith('.hea')} if os.path.exists(d) else set()
    return sorted(r for r in heas if os.path.exists(os.path.join(d, r + '.dat')) and os.path.exists(os.path.join(d, r + '.atr')))

for db in EXT_DBS:
    d = os.path.join(EXT_DIR, db)
    if len(complete_records(d)) < EXT_DBS[db]['n_expected']:
        os.makedirs(d, exist_ok=True); print('Đang tải', db, 'từ PhysioNet ...'); wfdb.dl_database(db, dl_dir=d)
    EXT_DBS[db]['records'] = complete_records(d)
    assert len(EXT_DBS[db]['records']) == EXT_DBS[db]['n_expected'], f'Thiếu bản ghi {db}'
    print(f"{EXT_DBS[db]['name']}: {len(EXT_DBS[db]['records'])} bản ghi")

def incart_patients():
    # files-patients-diagnoses.txt: dòng 'patient N' rồi dòng liệt kê bản ghi 'I01 I02 ...'
    import re
    p = os.path.join(EXT_DIR, 'incartdb', 'files-patients-diagnoses.txt')
    if not os.path.exists(p):
        wfdb.dl_files('incartdb', os.path.join(EXT_DIR, 'incartdb'), ['files-patients-diagnoses.txt'])
    mp, cur = {}, None
    for line in open(p, encoding='latin-1'):
        m = re.match(r'\s*patient\s+(\d+)', line, re.I)
        if m:
            cur = int(m.group(1)); continue
        for r in re.findall(r'\bI\d{2}\b', line):
            if cur is not None:
                mp[f'incartdb/{r}'] = cur
    return mp

PATIENT = incart_patients()
assert len(PATIENT) == 75 and len(set(PATIENT.values())) == 32, f'Bảng bệnh nhân INCART: {len(PATIENT)} bản ghi / {len(set(PATIENT.values()))} bệnh nhân'
print('INCART: 75 bản ghi thuộc 32 bệnh nhân')

EXT = {}
for db in EXT_DBS:
    t0 = time.time()
    recs = [f'{db}/{r}' for r in EXT_DBS[db]['records']]
    X, R, y, g, T = build_dataset(recs)
    cal = pa.split_first(T, g, 5)
    EXT[db] = dict(X=X, R=R, y=y, g=g, T=T, cal=cal, ev=~cal)
    print(f"{EXT_DBS[db]['name']}: {len(y)} nhịp, {time.time() - t0:.0f} s")
    describe(y, '   toàn bộ'); describe(y[~cal], '   đánh giá (sau phút 5)')

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

def row_of(m, **kv):
    r = dict(kv, acc=m['acc'], macroF1=m['macroF1'], kappa=m['kappa'], jk=m['jk'])
    for c in pa.CLASSES:
        r[f'Se_{c}'] = m['Se'][c]; r[f'PP_{c}'] = m['PP'][c]
    return r

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for i, k in enumerate(o):
        run = max(run, (len(p) - i) * p[k]); adj[k] = min(run, 1.0)
    return adj

def wil(v, base):
    ok = ~np.isnan(v) & ~np.isnan(base); d = v[ok] - base[ok]
    return dict(n=int(ok.sum()), better=int((d > 1e-9).sum()), worse=int((d < -1e-9).sum()),
                median_gain=float(np.median(d)), p=float(wilcoxon(v[ok], base[ok]).pvalue))

COLS = ['acc', 'macroF1', 'kappa', 'jk', 'Se_N', 'PP_N', 'Se_S', 'PP_S', 'Se_V', 'PP_V', 'Se_F', 'PP_F']
RES = {}
for db, D in EXT.items():
    rows, rec_scores = [], {}
    for seed in SEEDS:
        S = SEED_DATA[seed]
        E = emb_cached(db, seed, D['X'], D['R'], S) / 2.0 ** S['fe']
        pa.CURRENT_ANCHORS = ANCHORS[seed]
        Wq, bq = pa.quantize_head(S['W0'], S['b0'], S['fw'])
        rows.append(row_of(pa.metrics(D['y'], (E @ Wq + bq).argmax(1)), seed=seed, method='none (all beats)'))
        for mth in METHODS:
            meth, hp = CONF[mth]
            yy, pp, rr = M.run(E, D['y'], D['g'], D['cal'], D['ev'], S['W0'], S['b0'], ANCHORS[seed], meth, fW=S['fw'], **hp)
            rows.append(row_of(pa.metrics(yy, pp), seed=seed, method=mth))
            rec_scores[(seed, mth)] = pa.per_record_scores(yy, pp, rr)
        print(db, 'seed', seed, 'xong')
    res = pd.DataFrame(rows); res.to_csv(os.path.join(OUT, f'ext_rev5c_{db}_gross.csv'), index=False)
    order = ['none (all beats)'] + METHODS
    summ = res.groupby('method')[COLS].agg(lambda v: f'{v.mean():.3f} ± {v.std():.3f}').loc[order]
    summ.to_csv(os.path.join(OUT, f'ext_rev5c_{db}_gross_summary.csv'))
    print(f"\n=== {EXT_DBS[db]['name']} — chỉ số gộp (trung bình ± SD 5 seed)"); display(summ)

    recs = sorted(set(D['g'][D['ev']]))
    tab = lambda mth, key: np.array([[rec_scores[(s, mth)][r][key] for r in recs] for s in SEEDS]).mean(0)
    per = pd.DataFrame({f'{m}|{k}': tab(m, k) for m in METHODS for k in ['balacc', 'Se_S', 'Se_V', 'Se_F']}, index=recs)
    per.to_csv(os.path.join(OUT, f'ext_rev5c_{db}_per_record.csv'))
    avg = pd.DataFrame([dict(method=m, **{k: np.nanmean(tab(m, k)) for k in ['acc', 'balacc', 'Se_N', 'Se_S', 'Se_V', 'Se_F']})
                        for m in METHODS]).set_index('method')
    avg.to_csv(os.path.join(OUT, f'ext_rev5c_{db}_average_stats.csv'))

    base = tab('none', 'balacc'); w = []
    for m in METHODS[1:]:
        w.append(dict(method=m, unit=f'record (n={len(recs)})', **wil(tab(m, 'balacc'), base)))
    if db == 'incartdb':                      # 75 bản ghi không độc lập -> thêm mức bệnh nhân
        pid = np.array([PATIENT[r] for r in recs]); up = np.unique(pid)
        pat = lambda v: np.array([np.nanmean(v[pid == u]) for u in up])
        for m in METHODS[1:]:
            w.append(dict(method=m, unit=f'patient (n={len(up)})', **wil(pat(tab(m, 'balacc')), pat(base))))
    wt = pd.DataFrame(w)
    wt['p_holm'] = np.nan
    for u in wt['unit'].unique():                 # Holm trong từng họ kiểm định (cùng đơn vị)
        k = wt['unit'] == u; wt.loc[k, 'p_holm'] = holm(wt.loc[k, 'p'].values)
    wt.to_csv(os.path.join(OUT, f'ext_rev5c_{db}_wilcoxon.csv'), index=False)
    print('=== Wilcoxon (balanced accuracy, trung bình 5 seed)'); display(wt.round(6))
    RES[db] = dict(gross=res, summ=summ, wil=wt)

In [ ]:
NAMES = {'incartdb': 'INCART', 'svdb': 'SVDB'}
summary = pd.concat({EXT_DBS[db]['name']: RES[db]['summ'].loc[METHODS, ['acc', 'macroF1', 'jk', 'Se_S', 'PP_S', 'Se_V', 'PP_V']]
                     for db in EXT}, axis=0)
summary.to_csv(os.path.join(OUT, 'ext_rev5c_summary_all.csv')); display(summary)

# Đối chiếu với pipeline fake-quant cũ (06_eval/external_incart_svdb) -> ghi chênh lệch trung bình
cmp_rows = []
for db in EXT:
    old_p = os.path.join(OLD_DIR, f'ext_{db}_gross.csv')
    if not os.path.exists(old_p):
        print('Không thấy kết quả cũ:', old_p); continue
    old = pd.read_csv(old_p).groupby('method').mean(numeric_only=True)
    new = RES[db]['gross'].groupby('method').mean(numeric_only=True)
    for m in ['none']:
        for k in ['acc', 'macroF1', 'jk', 'Se_S', 'Se_V']:
            cmp_rows.append(dict(db=NAMES[db], method=m, metric=k, old=old.loc[m, k], new=new.loc[m, k], diff=new.loc[m, k] - old.loc[m, k]))
cmp = pd.DataFrame(cmp_rows); cmp.to_csv(os.path.join(OUT, 'ext_int_vs_old_fakequant.csv'), index=False)
if len(cmp):
    print('Chênh lệch lớn nhất (mới - cũ):', cmp.loc[cmp['diff'].abs().idxmax()].to_dict())
    display(cmp.pivot_table(index=['db', 'method'], columns='metric', values='diff').round(4))

# Bảng LaTeX dán thẳng vào bản thảo (thay tab:ext)
def ms(v):
    return f'.{round(v.mean() * 1000):03d}$\\pm$.{round(v.std() * 1000):03d}'
lines = [r'\begin{tabular}{llccccc}', r'\toprule',
         r'Database & Method & Acc & mF1 & $j\kappa$ & Se$_S$ & Se$_V$\\', r'\midrule']
for j, db in enumerate(EXT):
    g = RES[db]['gross']
    for i, m in enumerate(METHODS):
        v = g[g.method == m]
        first = rf'\multirow{{{len(METHODS)}}}{{*}}{{{NAMES[db]}}}' if i == 0 else ''
        f3 = lambda x: f'{x:.3f}'.lstrip('0')
        mname = m.replace('_', '\\_')
        lines.append(f'{first} & {mname} & {ms(v.acc)} & {ms(v.macroF1)} & {ms(v.jk)} & {f3(v.Se_S.mean())} & {f3(v.Se_V.mean())}' + r'\\')
    if j == 0:
        lines.append(r'\midrule')
lines += [r'\bottomrule', r'\end{tabular}']
open(os.path.join(OUT, 'tab_ext_rev5c.tex'), 'w').write('\n'.join(lines) + '\n')
print('\n'.join(lines))

In [ ]:
# Số bản ghi bị hại theo bệnh nhân (balanced accuracy thấp hơn none quá 0.05, trung bình 5 seed) + đối chiếu với rev5b
rows = []
for db in EXT:
    per = pd.read_csv(os.path.join(OUT, f'ext_rev5c_{db}_per_record.csv'), index_col=0)
    for m in METHODS[1:]:
        d = per[f'{m}|balacc'] - per['none|balacc']
        rows.append(dict(db=NAMES[db], method=m, n_records=int(d.notna().sum()), better=int((d > 1e-9).sum()), worse=int((d < -1e-9).sum()),
                         harmed_005=int((d < -0.05).sum()), worst=float(d.min()), mean_balacc=float(per[f'{m}|balacc'].mean())))
harm = pd.DataFrame(rows); harm.to_csv(os.path.join(OUT, 'ext_rev5c_harm.csv'), index=False); display(harm)
old = pd.read_csv(os.path.join(OLD5B, 'ext_rev5b_summary_all.csv')); new = pd.read_csv(os.path.join(OUT, 'ext_rev5c_summary_all.csv'))
chk = old.merge(new, on=[old.columns[0], 'method'], suffixes=('_5b', '_5c'))
bad = [(r[old.columns[0]], r.method, c) for _, r in chk.iterrows() for c in ['acc', 'macroF1', 'jk', 'Se_S', 'PP_S', 'Se_V', 'PP_V'] if r[c + '_5b'] != r[c + '_5c']]
print(f'Đối chiếu rev5b: {len(chk)} dòng, {len(bad)} ô khác', bad[:5])
assert not bad, 'Phương pháp cũ cho số khác rev5b -> kiểm tra lại'
print('==> Các phương pháp cũ trùng rev5b. head_scratch_fair:'); display(new[new.method == 'head_scratch_fair'])

In [ ]:
import scipy, sklearn
log = dict(notebook='notebooks/ecg_ext_validation_rev5c_20261029.ipynb',
           run_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
           python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__,
           tensorflow=tf.__version__, wfdb=wfdb.__version__,
           input_sha256=HASHES, hyperparameters={k: v[1] for k, v in CONF.items()}, seeds=SEEDS,
           checks=dict(fast_vs_hw_ref='bit-exact (1500 DS2 beats x 5 seeds)',
                       board_seed3_DS2='0 mismatches (none, offset_int, gradient_anchor)',
                       ds2_none_acc_seed3=float(acc_all)),
           data={db: dict(records=len(EXT_DBS[db]['records']), beats=int(len(EXT[db]['y'])),
                          eval_beats=int(EXT[db]['ev'].sum()),
                          eval_counts=dict(zip(CLASS_NAMES, np.bincount(EXT[db]['y'][EXT[db]['ev']], minlength=4).tolist())))
                 for db in EXT},
           outputs=sorted(f for f in os.listdir(OUT)))
json.dump(log, open(os.path.join(OUT, 'run_log.json'), 'w'), indent=1, ensure_ascii=False)
_unused = (
'''# 06_eval/external_int — ngoại kiểm INCART + SVDB bằng pipeline số nguyên bit-exact
Sinh bởi notebooks/ecg_ext_validation_rev5c_20261029.ipynb. Thay cho 06_eval/external_incart_svdb (fake-quant, chỉ giữ để đối chiếu).
- Emb: hw_ref (AP_RND/AP_SAT); lớp đầu ra lượng tử theo thanh ghi head_w/head_b; siêu tham số exp4 (chọn trên DS1).
- Kiểm tra trước khi chạy: bản nhanh trùng bit hw_ref; seed 3 DS2 khớp board 0 nhịp lệch.
- ext_int_<db>_gross*.csv: chỉ số gộp từng seed / trung bình±SD; _per_record.csv; _average_stats.csv
- ext_int_<db>_wilcoxon.csv: theo bản ghi (+ theo bệnh nhân với INCART), p_holm = hiệu chỉnh Holm trong từng họ.
- ext_int_vs_old_fakequant.csv: chênh lệch so với pipeline cũ. tab_ext_int.tex: bảng dán vào bản thảo.
- run_log.json: phiên bản thư viện, SHA-256 mọi file đầu vào, số nhịp.
''')
print('Đã lưu vào', OUT); print(sorted(os.listdir(OUT)))